# 1. Data Cleaning

In [2]:
import pandas as pd

# Step 1: Directly load train and test datasets by name
train_path = 'train.csv'
test_path = 'test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Loaded train dataset: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Loaded test dataset: {test.shape[0]} rows, {test.shape[1]} columns")

Loaded train dataset: 8523 rows, 12 columns
Loaded test dataset: 5681 rows, 11 columns


In [3]:
# Step 2: Inspect Categorical Inconsistencies and Missing Values
print("Train Item_Fat_Content values:\n", train['Item_Fat_Content'].value_counts())
print("\nTrain Outlet_Size values:\n", train['Outlet_Size'].value_counts(dropna=False))

Train Item_Fat_Content values:
 Item_Fat_Content
Low Fat    5089
Regular    2889
LF          316
reg         117
low fat     112
Name: count, dtype: int64

Train Outlet_Size values:
 Outlet_Size
Medium    2793
NaN       2410
Small     2388
High       932
Name: count, dtype: int64


In [4]:
import numpy as np

def clean_dataset(df, outlet_size_fill='Small'):
    df_clean = df.copy()

    # 1. Standardize Item_Fat_Content
    df_clean['Item_Fat_Content'] = df_clean['Item_Fat_Content'].replace({
        'LF': 'Low Fat',
        'low fat': 'Low Fat',
        'reg': 'Regular'
    })

    # 2. Item_Weight

    # mean weight for each item identifier
    item_weight_map = df_clean.groupby('Item_Identifier')['Item_Weight'].transform('mean')
    df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(item_weight_map)

    # remaining missing weights using the global median
    global_median_weight = df_clean['Item_Weight'].median()
    df_clean['Item_Weight'] = df_clean['Item_Weight'].fillna(global_median_weight)


    # 3. Impute Item_Visibility

    # Replace zero values with NaN first
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].replace(0, np.nan)

    # Impute missing visibility using the same item's mean nonzero visibility
    item_visibility_map = df_clean.groupby('Item_Identifier')['Item_Visibility'].transform('mean')
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(item_visibility_map)

    # Impute any remaining missing visibility values using the global median of nonzero visibility
    global_median_visibility = df_clean['Item_Visibility'].median()
    df_clean['Item_Visibility'] = df_clean['Item_Visibility'].fillna(global_median_visibility)

    # 4. Handle missing Outlet_Size values (using 'Small' or 'Unknown' depending on mode)
    df_clean['Outlet_Size'] = df_clean['Outlet_Size'].fillna(outlet_size_fill)

    return df_clean

# Run the updated cleaning pipeline matching all criteria
train_clean = clean_dataset(train, outlet_size_fill='Small')
test_clean = clean_dataset(test, outlet_size_fill='Small')

# Save clean dataframes to separate new CSV files
train_clean.to_csv('train_cleaned.csv', index=False)
test_clean.to_csv('test_cleaned.csv', index=False)

print("Cleaning complete! Saved as 'train_cleaned.csv' and 'test_cleaned.csv'.\n")
print("Cleaned Train Missing Values:\n", train_clean.isnull().sum())
print("\nCleaned Test Missing Values:\n", test_clean.isnull().sum())

Cleaning complete! Saved as 'train_cleaned.csv' and 'test_cleaned.csv'.

Cleaned Train Missing Values:
 Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

Cleaned Test Missing Values:
 Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
dtype: int64
